# A/B Testing LLM Features — GrowthBook, Statsig, and the Vibes Problem: Interactive Visual Explorer

> Traditional A/B testing was not built for non-deterministic LLMs. The critical distinction: evals answer "can the model do the job?" A/B tests answer "do users care?" Both are required; shipping on vibe checks is over. What to test in 2026: prompt engineering (wording), model selection (GPT-4 vs GPT-3.5 vs OSS; accuracy vs cost vs latency), generation parameters (temperature, top-p). Real cases: a chatbot reward-model variant delivered +70% conversation length and +30% retention; Nextdoor AI subject-line experiments delivered +1% CTR after reward-function refinement; Khan Academy Khanmigo iterated on a latency-vs-math-accuracy axis. Platform split: **Statsig** (acquired by OpenAI for $1.1B in September 2025) — sequential testing, CUPED, all-in-one. **GrowthBook** — open-source, warehouse-native, Bayesian + Frequentist + Sequential engines, CUPED, SRM checks, Benjamini-Hochberg + Bonferroni corrections. You pick based on warehouse-SQL preference and whether "acquired by OpenAI" matters to your organization.

Welcome to the interactive companion notebook for **A/B Testing LLM Features — GrowthBook, Statsig, and the Vibes Problem**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""Sequential A/B test simulator — stdlib Python.

Compares fixed-sample vs always-valid sequential testing on a binary outcome.
Illustrates CUPED-style variance reduction.
"""

from __future__ import annotations

import math
import random

def z_statistic(success_a: int, n_a: int, success_b: int, n_b: int) -> float:
    p_a = success_a / n_a if n_a else 0
    p_b = success_b / n_b if n_b else 0
    p = (success_a + success_b) / (n_a + n_b) if (n_a + n_b) else 0
    se = math.sqrt(p * (1 - p) * (1 / n_a + 1 / n_b)) if n_a and n_b else 1
    return (p_b - p_a) / se if se > 0 else 0


In [ ]:
def fixed_sample_size(p_baseline: float, lift: float, alpha: float = 0.05, power: float = 0.80) -> int:
    p_treat = p_baseline * (1 + lift)
    z_alpha = 1.96
    z_beta = 0.84
    p_bar = (p_baseline + p_treat) / 2
    num = (z_alpha * math.sqrt(2 * p_bar * (1 - p_bar)) +
           z_beta * math.sqrt(p_baseline * (1 - p_baseline) + p_treat * (1 - p_treat))) ** 2
    den = (p_treat - p_baseline) ** 2
    return int(num / den)


In [ ]:
def simulate(p_a: float, p_b: float, seed: int = 7, max_n: int = 300_000) -> dict:
    rng = random.Random(seed)
    success_a = success_b = 0
    n_a = n_b = 0
    sequential_stopped_at = None
    for _ in range(max_n):
        group = rng.random() < 0.5
        if group:
            n_b += 1
            if rng.random() < p_b:
                success_b += 1
        else:
            n_a += 1
            if rng.random() < p_a:
                success_a += 1
        if n_a > 100 and n_b > 100 and sequential_stopped_at is None:
            z = z_statistic(success_a, n_a, success_b, n_b)
            # Always-valid z-boundary (mSPRT-style): grows with log(n) so Type-I stays bounded.
            # threshold(n) ≈ sqrt(2 * log(1/alpha) + log(n)) for alpha=0.05.
            n_total = n_a + n_b
            threshold = math.sqrt(2 * math.log(1 / 0.05) + math.log(n_total))
            if abs(z) > threshold:
                sequential_stopped_at = n_total
                break


In [ ]:
return {
        "n_a": n_a,
        "n_b": n_b,
        "p_a_observed": success_a / n_a if n_a else 0.0,
        "p_b_observed": success_b / n_b if n_b else 0.0,
        "sequential_stop_at": sequential_stopped_at,
    }

def main() -> None:
    print("=" * 80)
    print("SEQUENTIAL A/B — fixed vs always-valid, binary outcome")
    print("=" * 80)

baseline = 0.03
    for lift in (0.02, 0.05, 0.10):
        required = fixed_sample_size(baseline, lift)
        adjusted = int(required * 1.4)  # LLM non-determinism buffer
        print(f"\nBaseline {baseline*100:.0f}%, lift +{lift*100:.0f}%:")
        print(f"  fixed sample size (traditional, 80% power, α=0.05): {required}")
        print(f"  LLM-adjusted (×1.4 for non-determinism): {adjusted}")


In [ ]:
print("\nSimulation — actual lift 5% (p_a=0.03, p_b=0.0315):")
    result = simulate(0.03, 0.0315)
    print(f"  final n: A={result['n_a']}, B={result['n_b']}")
    print(f"  observed: p_a={result['p_a_observed']*100:.3f}%, p_b={result['p_b_observed']*100:.3f}%")
    print(f"  sequential stop at n={result['sequential_stop_at']}")

print("\nSimulation — actual lift 10% (p_a=0.03, p_b=0.033):")
    result = simulate(0.03, 0.033)
    print(f"  final n: A={result['n_a']}, B={result['n_b']}")
    print(f"  observed: p_a={result['p_a_observed']*100:.3f}%, p_b={result['p_b_observed']*100:.3f}%")
    print(f"  sequential stop at n={result['sequential_stop_at']}")


In [ ]:
print("\nSimulation — actual lift 50% (p_a=0.03, p_b=0.045) — strong signal:")
    result = simulate(0.03, 0.045)
    print(f"  final n: A={result['n_a']}, B={result['n_b']}")
    print(f"  observed: p_a={result['p_a_observed']*100:.3f}%, p_b={result['p_b_observed']*100:.3f}%")
    print(f"  sequential stop at n={result['sequential_stop_at']}")

print("\nRead: on strong signals the sequential bound fires early (the 50% lift")
    print("case above), and the returned n_a/n_b reflect samples *up to* the stop")
    print("point, not the full horizon. For small or zero effects the bound is")
    print("deliberately conservative — that is the Type-I guarantee.")


In [ ]:
if __name__ == "__main__":
    main()
